## Step 1: Assemble Corpus

In [1]:
import re
import requests
from collections import Counter, defaultdict

import numpy as np
import time


In [2]:

DATASET_URL = "https://raw.githubusercontent.com/bhagatpandey369/NLP/refs/heads/main/dataset/1-18%20books%20combined.txt"
CORPUS_FILE = "mahabharata_corpus.txt"

In [3]:
response = requests.get(DATASET_URL)

response.raise_for_status()
corpus = response.text
print(corpus[:200])

Adi Parva
 
Chapter One
Maharaja Shantanu Marries the Celestial Ganga
 
According to the historical records of this earth, there once lived a King named Maharaja Shantanu, the son of Pratipa, who took


In [4]:
len(corpus.splitlines())

3118

In [5]:
len(corpus.split())

290922

In [6]:
len(corpus)

1701795

In [7]:
text = corpus.lower()
tokens = re.findall(r'\b\w+\b', text)
len(tokens)

293626

In [8]:
text[:100]

'adi parva\n \nchapter one\nmaharaja shantanu marries the celestial ganga\n \naccording to the historical '

In [9]:
tokens[:10]

['adi',
 'parva',
 'chapter',
 'one',
 'maharaja',
 'shantanu',
 'marries',
 'the',
 'celestial',
 'ganga']

In [10]:
word_counts = Counter(tokens)
len(word_counts)

10335

In [11]:
for word, count in word_counts.most_common(20):
    print(word, '-->', count)

the --> 20561
of --> 10737
and --> 9651
to --> 7874
in --> 4369
his --> 3962
a --> 3469
he --> 3294
with --> 3182
that --> 2976
you --> 2925
is --> 2785
i --> 2632
was --> 2545
by --> 2308
this --> 2211
s --> 2127
all --> 1881
arjuna --> 1881
for --> 1858


## Step 2 — Define Vocabulary Size (M)

In [16]:
M = 1000

most_common_words = word_counts.most_common(M)
most_common_words[-10:]

[('blind', 34),
 ('assist', 34),
 ('aside', 34),
 ('reactions', 34),
 ('attachment', 34),
 ('meant', 34),
 ('conchshells', 34),
 ('witnessing', 34),
 ('pilgrimage', 34),
 ('vikarna', 34)]

In [17]:
vocab = [word for word, count in most_common_words]
vocab[:10]

['the', 'of', 'and', 'to', 'in', 'his', 'a', 'he', 'with', 'that']

In [18]:
len(vocab)

1000

In [19]:
word_to_index = {word: idx for idx, word in enumerate(vocab)}


In [20]:
for word in vocab[:10]:
    print(word, '-->', word_to_index[word])

the --> 0
of --> 1
and --> 2
to --> 3
in --> 4
his --> 5
a --> 6
he --> 7
with --> 8
that --> 9


In [21]:
idx_to_word = {idx: word for word, idx in word_to_index.items()}
for idx in range(10):
    print(idx, '-->', idx_to_word[idx])

0 --> the
1 --> of
2 --> and
3 --> to
4 --> in
5 --> his
6 --> a
7 --> he
8 --> with
9 --> that


In [22]:
len(tokens)

293626

In [23]:
corpus_ids = np.array([word_to_index[word] for word in tokens if word in word_to_index], dtype=np.int32)

In [24]:
len(corpus_ids)

246736

In [25]:
corpus_ids

array([354, 148,  42, ..., 660, 417, 417], dtype=int32)

In [26]:
round(len(corpus_ids) / len(tokens) *100,2)

84.03

In [27]:
for idx in corpus_ids[:10]:
    print(idx_to_word[idx], end=" ")

parva chapter one maharaja shantanu the celestial ganga according to 

## Step 3 — Choose Context Window (C)

In [28]:
# Context window

C = 2

window_size= 2 * C + 1
window_size

5

In [29]:
example_words = [idx_to_word[idx] for idx in corpus_ids[:10]]
' '.join(example_words)

'parva chapter one maharaja shantanu the celestial ganga according to'

In [30]:
for i in range(C, C+3):
    target_id = corpus_ids[i]
    target_word = idx_to_word[target_id]

    context = []

    for j in range(i-C, i + C +1):
        if j != i:
            context.append(idx_to_word[corpus_ids[j]])

    print(target_word, '-->', context)    

one --> ['parva', 'chapter', 'maharaja', 'shantanu']
maharaja --> ['chapter', 'one', 'shantanu', 'the']
shantanu --> ['one', 'maharaja', 'the', 'celestial']


## Step 4: Build Co-occurrence Dictionary

In [31]:
cooccurrence = defaultdict(Counter)
cooccurrence

defaultdict(collections.Counter, {})

In [32]:
for i in range(C, len(corpus_ids) - C):

    traget = int(corpus_ids[i])

    for j in range(i-C, i+C+1):
        if i == j:
            continue

        context = int(corpus_ids[j])
        cooccurrence[traget][context] += 1

In [33]:
cooccurrence[100].most_common(5)

[(0, 155), (5, 140), (2, 129), (445, 71), (3, 58)]

In [34]:
for traget_id in range(5):
    traget_word = idx_to_word[traget_id]

    context_words = [(idx_to_word[context_id], count) for context_id, count in cooccurrence[traget_id].most_common(5)]

    print(traget_word, '-->', context_words)

the --> [('of', 9170), ('the', 3672), ('and', 3229), ('to', 2747), ('in', 1943)]
of --> [('the', 9170), ('and', 1304), ('a', 774), ('son', 692), ('all', 577)]
and --> [('the', 3229), ('of', 1304), ('his', 1018), ('to', 938), ('and', 624)]
to --> [('the', 2747), ('and', 938), ('his', 613), ('be', 569), ('him', 553)]
in --> [('the', 1943), ('and', 550), ('of', 483), ('this', 402), ('to', 371)]


In [35]:
def show_cooccurrences(word, tok_k):

    if word not in word_to_index:
        return "Word not found in vocabulary."

    target_id = word_to_index[word]

    print('Target word', word)
    print('-'*10)

    for context_id, count in cooccurrence[target_id].most_common(tok_k):
        print(idx_to_word[context_id],'-->', count)


In [36]:
show_cooccurrences('krishna', 5)

Target word krishna
----------
lord --> 911
the --> 309
and --> 280
of --> 230
to --> 174


In [37]:
show_cooccurrences('apple', 5)

'Word not found in vocabulary.'

## Step 5: Choose Embedding Size (N)

In [38]:
# Embedding Dimension
N = 100

print('Embedding Matrix:')
M , '*' , N

Embedding Matrix:


(1000, '*', 100)

## Step 6: Initialize Two Tables (E: Target words and U: Context Word)

In [39]:
np.random.seed(42)

# random initialization
E = np.random.normal(loc=0.0, scale=0.01, size=(M, N)).astype(np.float32)
U = np.random.normal(loc=0.0, scale=0.01, size=(M, N)).astype(np.float32)


In [40]:
E.shape

(1000, 100)

In [41]:
U.shape

(1000, 100)

In [46]:
word = vocab[100]
word_id = word_to_index[word]
print('word ', word)
print('ID ', word_id)
print('Embedding word ', E[word_id][:5])

word  bow
ID  100
Embedding word  [-0.00678495 -0.00305499 -0.00597381  0.00110418  0.01197179]


## Step 7 — Train Embeddings with SGNS

In [47]:
learning_rate = 0.025
negative_samples = 5
epoch = 5

In [48]:
vocab_counts = np.array([word_counts[word] for word in vocab], dtype=np.float64)
vocab_counts[:20]

array([20561., 10737.,  9651.,  7874.,  4369.,  3962.,  3469.,  3294.,
        3182.,  2976.,  2925.,  2785.,  2632.,  2545.,  2308.,  2211.,
        2127.,  1881.,  1881.,  1858.])

In [49]:
negative_distribution = vocab_counts ** 0.75
negative_distribution = negative_distribution / negative_distribution.sum()
negative_distribution[:20]

array([0.03474598, 0.02134438, 0.01970383, 0.01691484, 0.01087447,
       0.01010549, 0.00914691, 0.00879861, 0.00857327, 0.00815354,
       0.00804852, 0.00775783, 0.00743594, 0.00725082, 0.00673827,
       0.00652474, 0.00633792, 0.0057798 , 0.0057798 , 0.00572672])

In [50]:
negative_distribution.sum()

np.float64(0.9999999999999999)

In [51]:
def sigmoid(x):
    x = np.clip(x, -15, 15)
    x = 1.0 / (1.0 + np.exp(-x))
    return x

In [52]:
values = np.array([-5, -2, 0, 2, 5])
sigmoid(values)

array([0.00669285, 0.11920292, 0.5       , 0.88079708, 0.99330715])

In [53]:
len(corpus_ids)

246736

In [54]:
def generate_context_pairs(corpus_ids, window_size):
    n = len(corpus_ids)

    for i in range(n):

        target = int(corpus_ids[i])

        start = max(0, i - window_size)
        end = min(n, i + window_size + 1)

        for j in range(start, end):

            if i == j:
                continue

            context = int(corpus_ids[j])

            yield target, context

In [55]:
pair_generator = generate_context_pairs(corpus_ids[:10], C)

for target, context in list(pair_generator)[:10]:

    print(f"{idx_to_word[target]:15s} -> "f"{idx_to_word[context]}")

parva           -> chapter
parva           -> one
chapter         -> parva
chapter         -> one
chapter         -> maharaja
one             -> parva
one             -> chapter
one             -> maharaja
one             -> shantanu
maharaja        -> chapter


In [56]:
def train_sgns(corpus_ids, E, U, negative_distribution, 
               window_size=2, negative_samples=5, learning_rate=0.025,
               epochs=3, max_tokens=None):
    if max_tokens is not None:
        train_corpus = corpus_ids[:max_tokens]

    else:
        train_corpus = corpus_ids

    total_pairs_per_epoch = 0

    for epoch in range(epochs):
        start_time = time.time()

        total_loss = 0.0
        pair_count = 0

        n = len(train_corpus)

        for i in range(n):
            target_id = int(train_corpus[i])

            start = max(0, i - window_size)
            end = min(n, i+window_size+1)

            target_vector = E[target_id].copy()

            for j in range(start, end):
                if i == j:
                    continue

                context_id = int(train_corpus[j])


                # positive sample
                context_vector = U[context_id].copy()
                score = np.dot(target_vector, context_vector)
                prediction = sigmoid(score)
                error = prediction - 1
                total_loss = total_loss - np.log(max(prediction, 1e-10))

                grad_target = error * context_vector
                grad_context = error * target_vector

                E[target_id] = E[target_id] - learning_rate * grad_target
                U[context_id] = U[context_id] - learning_rate * grad_context

                #negative samples

                negative_ids = np.random.choice(len(negative_distribution), size=negative_samples, p=negative_distribution)

                for negative_id in negative_ids:
                    negative_id = int(negative_id)

                    if negative_id == context_id:
                        continue

                    negative_vector = U[negative_id].copy()

                    negative_score = np.dot(target_vector, negative_vector)
                    negative_prediction = sigmoid(negative_score)

                    negative_error = negative_prediction

                    total_loss = total_loss - np.log(max(1 - negative_prediction, 1e-10))

                    grad_target_negative = negative_error * negative_vector
                    grad_negative = negative_error * target_vector

                    E[target_id] = E[target_id] - (learning_rate * grad_target_negative)
                    U[negative_id] = U[negative_id] - (learning_rate * grad_negative)

                pair_count = pair_count + 1

        total_time = time.time() - start_time
        average_loss = total_loss / max(pair_count,1)
        print(f"Epoch {epoch + 1}/{epochs} | "f"Pairs: {pair_count:,} | "f"Loss: {average_loss:.4f} | "f"Time: {total_time:.2f}s")

    return E, U





        
    

In [57]:
E, U = train_sgns(
    corpus_ids=corpus_ids,
    E=E,
    U=U,
    negative_distribution=negative_distribution,
    window_size=C,
    negative_samples=negative_samples,
    learning_rate=learning_rate,
    epochs=2,
    max_tokens= None
)

Epoch 1/2 | Pairs: 986,938 | Loss: 2.4048 | Time: 410.12s
Epoch 2/2 | Pairs: 986,938 | Loss: 2.2486 | Time: 414.40s


In [58]:
E.shape

(1000, 100)

In [59]:
word = 'arjuna'
if word in word_to_index:
    word_index = word_to_index[word]
    print("Word ID:", word_id)
    print("\nLearned embedding:")
    print(E[word_id][:5])

Word ID: 100

Learned embedding:
[-0.49032423  0.13049015  0.25649366 -0.01908481 -0.35934603]


In [67]:
def cosine_similarity(vec1, vec2):

    denominator = (np.linalg.norm(vec1) * np.linalg.norm(vec2))

    if denominator == 0:
        return 0.0
    return np.dot(vec1, vec2) / denominator

In [72]:
def most_similar(word, E, word_to_index, idx_to_word, top_k=10):

    if word not in word_to_index:
        return 'Word not found in vocabulary.'

    word_id  = word_to_index[word]
    traget_vector = E[word_id]

    norms = np.linalg.norm(E, axis=1, keepdims=True)
    normalized_E = E / np.maximum(norms, 1e-10)

    target_normalized = (traget_vector / max(np.linalg.norm(traget_vector), 1e-10))

    similarities = normalized_E @ target_normalized
    similarities[word_id] = -np.inf

    top_indices = np.argsort(similarities)[-top_k:][::-1]
    print(f"\nWords most similar to '{word}':\n")

    for idx in top_indices:
        print(
            f"{idx_to_word[idx]:20s} "
            f"{similarities[idx]:.4f}"
        )

In [73]:
most_similar("arjuna", E, word_to_index, idx_to_word)


Words most similar to 'arjuna':

kichaka              0.6631
uttara               0.6597
shalva               0.6596
parashurama          0.6588
vyasadeva            0.6565
partha               0.6428
kuvera               0.6324
balarama             0.6218
conchshell           0.6195
satyaki              0.6161


In [74]:
most_similar("king", E, word_to_index, idx_to_word)


Words most similar to 'king':

maharaja             0.6838
monarch              0.6558
brahmana             0.6236
princess             0.6160
narada               0.6007
vyasa                0.5904
sanjaya              0.5895
sage                 0.5742
suta                 0.5732
queen                0.5721


In [75]:
most_similar("war", E, word_to_index, idx_to_word)


Words most similar to 'war':

formation            0.8715
river                0.8343
young                0.7878
fact                 0.7814
future               0.7806
lake                 0.7806
kshatriyas           0.7797
ceremony             0.7787
learned              0.7763
distance             0.7717
